# 사전 실험 A: 비트레이트 단계별 YOLO26s 성능 (UVG-VCM, 720p·30fps)
- Input: 데이터셋 구축 노트북 Output (`uvg_cctv`)
- 설정: Accelerator **GPU T4**, Internet **On** (yolo26s.pt 다운로드), **Save & Run All**
- 탐지기: YOLO26s COCO 사전학습 가중치 그대로 (학습 없음), 입력 640, conf 0.001, 이미지당 300개
- 평가: pycocotools, 원본(ref)·4 / 2 / 1 / 0.5 Mbps 각각. 전체 클래스 + CCTV 클래스(사람·자전거·자동차·오토바이·버스·트럭)
- 끝나면 Logs + Output의 `results_*.csv`, `overlay_*.png` 전달

In [ ]:
!pip install -q ultralytics==8.4.165 pycocotools

In [ ]:
%%writefile eval_lib.py
"""UVG-VCM 비트레이트 단계별 YOLO 평가용 함수"""
import json, os, subprocess, collections
import numpy as np

W, H = 1280, 720
LEVELS = ["ref", "4000k", "2000k", "1000k", "500k"]
CCTV_CLASSES = [1, 2, 3, 4, 6, 8]  # person, bicycle, car, motorcycle, bus, truck (class_id = COCO 순서 + 1)


def build_gt(raw, n_ref_frames, coco_names):
    """UVG-VCM 원본 라벨 -> 720p·30fps COCO GT
    원본 키 k(1부터) = 원본 프레임 k-1, 기준 영상 프레임 i = 원본 프레임 2i -> 키 2i+1"""
    images, anns, aid = [], [], 0
    for i in range(n_ref_frames):
        images.append({"id": i, "file_name": f"{i:06d}.png", "width": W, "height": H})
        for o in raw.get(str(2 * i + 1), []):
            x1, y1 = o["x_min"] * W, o["y_min"] * H
            x2, y2 = o["x_max"] * W, o["y_max"] * H
            w, h = x2 - x1, y2 - y1
            if w <= 0 or h <= 0:
                continue
            aid += 1
            anns.append({"id": aid, "image_id": i, "category_id": int(o["class_id"]),
                         "bbox": [round(x1, 2), round(y1, 2), round(w, 2), round(h, 2)],
                         "area": round(w * h, 2), "iscrowd": int(o.get("iscrowd", 0)),
                         "track_id": o.get("track_id")})
    cats = [{"id": i + 1, "name": n} for i, n in enumerate(coco_names)]
    return {"images": images, "annotations": anns, "categories": cats}


def gt_stats(gt, classes=None):
    a = [x for x in gt["annotations"] if classes is None or x["category_id"] in classes]
    s = sum(x["area"] < 32 ** 2 for x in a); m = sum(32 ** 2 <= x["area"] < 96 ** 2 for x in a)
    names = {c["id"]: c["name"] for c in gt["categories"]}
    cnt = collections.Counter(names[x["category_id"]] for x in a)
    return dict(boxes=len(a), small=s, medium=m, large=len(a) - s - m,
                small_ratio=round(s / max(1, len(a)), 3), top=dict(cnt.most_common(5)))


def extract_frames(video, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    r = subprocess.run(f"ffmpeg -y -loglevel error -i \"{video}\" -start_number 0 \"{out_dir}/%06d.png\"",
                       shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stderr[-2000:])
    return sorted(os.listdir(out_dir))


def filter_preds(preds, max_det=300, conf=0.001):
    by = collections.defaultdict(list)
    for p in preds:
        if p["score"] >= conf:
            by[p["image_id"]].append(p)
    out = []
    for v in by.values():
        v.sort(key=lambda x: -x["score"]); out.extend(v[:max_det])
    return out


def coco_eval(gt, preds, classes=None, img_ids=None):
    """pycocotools bbox 평가. classes: 평가할 category_id 목록 (None이면 GT에 있는 전체)"""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import contextlib, io
    g = COCO(); g.dataset = gt
    with contextlib.redirect_stdout(io.StringIO()):
        g.createIndex()
    present = sorted({a["category_id"] for a in gt["annotations"]})
    cat_ids = [c for c in present if classes is None or c in classes]
    ps = [p for p in filter_preds(preds) if p["category_id"] in cat_ids]
    if not ps or not cat_ids:
        return None
    with contextlib.redirect_stdout(io.StringIO()):
        d = g.loadRes(ps)
        E = COCOeval(g, d, "bbox")
        E.params.catIds = cat_ids
        if img_ids is not None:
            E.params.imgIds = img_ids
        E.evaluate(); E.accumulate(); E.summarize()
    names = ["AP", "AP50", "AP75", "APs", "APm", "APl", "AR1", "AR10", "AR100", "ARs", "ARm", "ARl"]
    return {k: round(float(v) * 100, 2) for k, v in zip(names, E.stats)}


In [ ]:
import os, sys, json, glob, shutil, time
import numpy as np, pandas as pd, torch
sys.path.insert(0, "/kaggle/working"); import eval_lib as E
from ultralytics import YOLO
assert torch.cuda.is_available(), "GPU 없음 -> Accelerator GPU T4"
cands = glob.glob("/kaggle/input/**/uvg_cctv", recursive=True); assert cands, "Input에 uvg_cctv 없음"
ROOT = cands[0]; OUT = "/kaggle/working"; TMP = "/tmp/frames"
SEQS = ["HighwayView", "JobFair", "VolleyballGame", "FloorballGame", "FloorballTrain"]   # 고정 시점
INCLUDE_CAMPUS = True   # CampusView는 따로 표기해서 같이 평가
if INCLUDE_CAMPUS: SEQS.append("CampusView")
model = YOLO("yolo26s.pt"); NAMES = [model.names[i] for i in range(80)]
print("ROOT", ROOT, "| classes", len(NAMES), NAMES[:4])

In [ ]:
GT = {}
for s in SEQS:
    meta = json.load(open(f"{ROOT}/{s}/done.json"))
    raw = json.load(open(f"{ROOT}/{s}/ann_raw.json"))
    GT[s] = E.build_gt(raw, meta["ref_frames"], NAMES)
    json.dump(GT[s], open(f"{OUT}/gt_{s}_720p30.json", "w"))
    print(f"{s:15s} frames {meta['ref_frames']} | 전체 {E.gt_stats(GT[s])} | CCTV {E.gt_stats(GT[s], E.CCTV_CLASSES)}")

In [ ]:
# 라벨-프레임 대응 확인: 기준 영상 0번 프레임에 GT(키 1) 박스를 그림 -> 객체에 맞으면 OK
from PIL import Image, ImageDraw
for s in ["HighwayView", "JobFair", "VolleyballGame"]:
    im = Image.open(f"{ROOT}/{s}/first.png").convert("RGB"); d = ImageDraw.Draw(im)
    for a in GT[s]["annotations"]:
        if a["image_id"] != 0: continue
        x, y, w, h = a["bbox"]; d.rectangle([x, y, x + w, y + h], outline=(0, 255, 0), width=2)
        d.text((x, max(0, y - 10)), NAMES[a["category_id"] - 1], fill=(255, 255, 0))
    im.save(f"{OUT}/overlay_{s}.png")
print("overlay saved")

In [ ]:
PRED = {}   # (seq, level) -> list
t0 = time.time()
for s in SEQS:
    for lv in E.LEVELS:
        video = f"{ROOT}/{s}/ref_720p30.mkv" if lv == "ref" else f"{ROOT}/{s}/h264_{lv}.mp4"
        fd = f"{TMP}/{s}_{lv}"; files = E.extract_frames(video, fd)
        assert len(files) == len(GT[s]["images"]), f"{s} {lv}: 프레임 {len(files)} != GT {len(GT[s]['images'])}"
        preds = []
        for i in range(0, len(files), 16):
            chunk = files[i:i + 16]
            res = model.predict([f"{fd}/{f}" for f in chunk], imgsz=640, conf=0.001, max_det=300,
                                device=0, verbose=False)
            for f, r in zip(chunk, res):
                iid = int(f[:6]); b = r.boxes
                for (x1, y1, x2, y2), c, k in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()):
                    preds.append({"image_id": iid, "category_id": int(k) + 1,
                                  "bbox": [round(float(x1), 2), round(float(y1), 2), round(float(x2 - x1), 2), round(float(y2 - y1), 2)],
                                  "score": round(float(c), 5)})
        PRED[(s, lv)] = preds
        json.dump(preds, open(f"{OUT}/pred_{s}_{lv}.json", "w"))
        shutil.rmtree(fd)
        print(f"{s:15s} {lv:6s} preds {len(preds):6d}  ({time.time()-t0:.0f}s)")

In [ ]:
def merged(seqs, lv):
    """여러 시퀀스를 하나의 GT/예측으로 합침 (image_id에 시퀀스별 오프셋)"""
    g = {"images": [], "annotations": [], "categories": GT[seqs[0]]["categories"]}; p = []; aid = 0
    for j, s in enumerate(seqs):
        off = j * 100000
        g["images"] += [{**im, "id": im["id"] + off} for im in GT[s]["images"]]
        for a in GT[s]["annotations"]:
            aid += 1; g["annotations"].append({**a, "id": aid, "image_id": a["image_id"] + off})
        p += [{**x, "image_id": x["image_id"] + off} for x in PRED[(s, lv)]]
    return g, p

FIXED = [s for s in SEQS if s != "CampusView"]
rows = []
for group, seqs in [("고정시점5", FIXED)] + [(s, [s]) for s in SEQS]:
    for cls_name, cls in [("전체클래스", None), ("CCTV클래스", E.CCTV_CLASSES)]:
        for lv in E.LEVELS:
            g, p = merged(seqs, lv)
            r = E.coco_eval(g, p, classes=cls)
            if r: rows.append({"group": group, "classes": cls_name, "level": lv, **r})
df = pd.DataFrame(rows).replace(-100.0, np.nan)   # 해당 크기 객체가 없으면 -100 -> 빈칸
df.to_csv(f"{OUT}/results_all.csv", index=False)
show = ["AP", "AP50", "APs", "APm", "APl"]
for (grp, cn), sub in df.groupby(["group", "classes"], sort=False):
    t = sub.set_index("level")[show]
    ref = t.loc["ref"] if "ref" in t.index else None
    print(f"\n=== {grp} / {cn} ===")
    print(t.to_string())
    if ref is not None:
        print("원본 대비 비율(%):"); print((t / ref.replace(0, np.nan) * 100).round(1).to_string())

In [ ]:
# 시퀀스별 실측 비트레이트·bpp 같이 정리
meta_rows = []
for s in SEQS:
    m = json.load(open(f"{ROOT}/{s}/done.json"))
    meta_rows.append({"seq": s, **{f"{k}_kbps": m[f"{k}_kbps"] for k in ["4000k", "2000k", "1000k", "500k"]},
                      **{f"{k}_bpp": m[f"{k}_bpp"] for k in ["4000k", "2000k", "1000k", "500k"]}})
pd.DataFrame(meta_rows).to_csv(f"{OUT}/results_bitrate.csv", index=False)
print(pd.DataFrame(meta_rows).to_string())